# Day 5：Python × 数据分析实践
## 上午统一课堂 Notebook（09:00–12:00）

**Day 2 你已经学会了这条分析链的金融逻辑**：

> 美国长端利率 → 30 年期抵押贷款利率 → 新屋销售 → 房地产 → 电动工具需求 → 创科实业 0669.HK / 泉峰控股 2285.HK

**今天不重新学这些金融知识。今天只回答一个问题：怎样用 Python 把同一套分析自动化，并且做成一个别人也能打开的产品？**

---

### 上午结构

| 时间 | 内容 |
|---|---|
| 09:00 | Part 0–1　配置与连接 |
| 09:20 | Part 2　取数：三种数据源，一套代码 |
| 09:50 | Part 3　Just Enough Pandas |
| 10:20 | ☕ 休息 |
| 10:35 | Part 4　可视化 |
| 11:00 | Part 5–6　股价与公司截面 |
| 11:25 | Part 7　三种数据结构的心智模型 |
| 11:35 | **Part 8　导出数据 ← 下午产品的起点** |
| 11:50 | Part 9　Streamlit 预览 |

### 下午你要做出来的东西

一个**公开的 GitHub 仓库**，里面有你的 Notebook、你导出的数据、一个 Streamlit 应用，以及一个部署好的网址。这个链接可以直接写进简历。

---
# Part 0 — 课堂配置

## 三种数据源，同一套分析代码

这是今天最重要的一个设计思想：**取数**和**分析**要分开。

只要把数据整理成同样形状的 DataFrame，后面所有的清洗、计算、画图代码**一行都不用改**。

| `DATA_SOURCE` | 数据从哪来 | 什么时候用 |
|---|---|---|
| `"wind"` | Wind 终端（需要 WindPy 且已登录） | 上午课堂演示 |
| `"fred"` | 本地 `fred_csv/` 文件夹里的 CSV | **下午做公开项目时用这个** |
| `"demo"` | 程序生成的模拟数据 | 没有 Wind、也还没下载 CSV 时 |

> 下午的公开仓库请使用 `"fred"`。FRED 的数据可以公开分享，只要在 README 里注明来源。

In [ ]:
# ===== 课堂配置 =====

DATA_SOURCE = "demo"        # "wind" / "fred" / "demo"

START_DATE = "2015-01-01"
END_DATE   = "2026-08-31"

# --- 如果 DATA_SOURCE = "wind"，把 Day 2 的 Wind Code Generator 结果填进来 ---
EDB_CODES = {
    "US_10Y_Treasury":   "PASTE_TREASURY_CODE_HERE",
    "US_30Y_Mortgage":   "PASTE_MORTGAGE_CODE_HERE",
    "US_New_Home_Sales": "PASTE_HOME_SALES_CODE_HERE",
    "US_CPI":            "PASTE_CPI_CODE_HERE",
}

# --- 如果 DATA_SOURCE = "fred"，下午从 FRED 下载这四个序列 ---
FRED_SERIES = {
    "US_10Y_Treasury":   "DGS10",         # 10 年期国债收益率（日频，美联储）
    "US_30Y_Mortgage":   "MORTGAGE30US",  # 30 年期固定房贷利率（周频，Freddie Mac）
    "US_New_Home_Sales": "HSN1F",         # 新屋销售（月频，Census，公共领域）
    "US_CPI":            "CPIAUCSL",      # CPI（月频，BLS）
}
FRED_DIR = "fred_csv"       # 放下载好的 CSV 的文件夹

# --- 股票（两家电动工具公司）---
SECURITIES  = ["0669.HK", "2285.HK"]
SEC_LABELS  = {"0669.HK": "Techtronic", "2285.HK": "Chervon"}
WSS_FIELDS  = ["pe_ttm", "pb_mrq", "mkt_cap_ard"]
WSS_LABELS  = ["PE(TTM)", "PB(MRQ)", "Market Cap"]

print("数据源：", DATA_SOURCE)

---
# Part 1 — 导入库

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 只有在需要时才尝试连接 Wind
WIND_AVAILABLE = False
if DATA_SOURCE == "wind":
    try:
        from WindPy import w
        w.start()
        WIND_AVAILABLE = w.isconnected()
        print("Wind 连接状态：", WIND_AVAILABLE)
    except Exception as e:
        print("WindPy 不可用，将回退到 demo 数据：", e)

print("pandas", pd.__version__)

---
# Part 2 — 取数：三种来源，同一个函数

下面这一段是**唯一**和数据源有关的代码。读一遍就好，不需要背。

重点看最后一行：不管数据从哪来，我们都得到一个 **DataFrame，行是日期，列是指标**。

In [ ]:
# ---------- 来源 A：Wind ----------
def _load_wind(name, code, start, end):
    if str(code).startswith("PASTE_") or not WIND_AVAILABLE:
        raise RuntimeError(f"{name}: Wind 不可用或代码未填写")
    r = w.edb(code, start, end, "")
    if r.ErrorCode != 0:
        raise RuntimeError(f"{name}: Wind ErrorCode={r.ErrorCode}")
    return pd.Series(r.Data[0], index=pd.to_datetime(r.Times), name=name).sort_index()


# ---------- 来源 B：FRED 下载的 CSV ----------
def _load_fred(name, series_id, folder=FRED_DIR):
    """读取从 fred.stlouisfed.org 下载的 CSV。"""
    path = Path(folder) / f"{series_id}.csv"
    if not path.exists():
        raise FileNotFoundError(f"找不到 {path}")

    df = pd.read_csv(path)
    date_col, value_col = df.columns[0], df.columns[1]   # FRED 只有两列
    dates = pd.to_datetime(df[date_col])
    # FRED 用 "." 表示缺失，errors="coerce" 会把它变成 NaN
    values = pd.to_numeric(df[value_col], errors="coerce")
    return pd.Series(values.values, index=dates, name=name).sort_index()


# ---------- 来源 C：模拟数据 ----------
def _load_demo(name, start, end):
    rng = np.random.default_rng(20260831)
    dates = pd.date_range(start, end, freq="ME")
    n = len(dates)
    t = np.linspace(0, 1, n)
    treasury = 2.0 + 1.8*t + 0.9*np.sin(np.linspace(0, 10, n)) + rng.normal(0, .12, n)
    mortgage = treasury + 2.4 + rng.normal(0, .18, n)
    sales    = 620 - 38*mortgage + 30*np.sin(np.linspace(0, 7, n)) + rng.normal(0, 15, n)
    cpi      = 250 + 60*t + 8*np.sin(np.linspace(0, 6, n))
    table = {"US_10Y_Treasury": treasury, "US_30Y_Mortgage": mortgage,
             "US_New_Home_Sales": sales, "US_CPI": cpi}
    return pd.Series(table.get(name, rng.normal(100, 5, n)), index=dates, name=name)


# ---------- 统一入口 ----------
def load_macro(names, source=None):
    source = source or DATA_SOURCE
    out = {}
    for name in names:
        try:
            if source == "wind":
                s = _load_wind(name, EDB_CODES[name], START_DATE, END_DATE)
            elif source == "fred":
                s = _load_fred(name, FRED_SERIES[name])
            else:
                s = _load_demo(name, START_DATE, END_DATE)
            tag = source.upper()
        except Exception as e:
            print(f"  [回退到 DEMO] {name}：{e}")
            s = _load_demo(name, START_DATE, END_DATE)
            tag = "DEMO"
        out[name] = s
        print(f"  [{tag}] {name}: {len(s)} 个观测值，{s.index.min().date()} 到 {s.index.max().date()}")
    return out

In [ ]:
NAMES = ["US_10Y_Treasury", "US_30Y_Mortgage", "US_New_Home_Sales", "US_CPI"]

series_dict = load_macro(NAMES)

# 把几个 Series 合并成一个 DataFrame。不同频率会自动按日期对齐，缺的地方是 NaN。
macro_raw = pd.concat(series_dict.values(), axis=1)
macro_raw = macro_raw.loc[START_DATE:END_DATE]

print()
print("形状：", macro_raw.shape)
macro_raw.tail()

### 停下来看一眼

上面这个 DataFrame 里有很多 `NaN`。**这是正常的，不是出错。**

原因是四个指标的频率不一样：国债收益率是日频，房贷利率是周频，新屋销售和 CPI 是月频。
`pd.concat` 按日期对齐，某一天只有日频数据有值，其他三列自然是空的。

下一步就是把它们统一到同一个频率。

---
# Part 3 — Just Enough Pandas

今天真正需要的只有这几个：

| 代码 | 作用 |
|---|---|
| `df.head()` / `df.tail()` | 看前几行 / 后几行 |
| `df.shape` / `df.columns` | 形状 / 列名 |
| `df.isna().sum()` | 每列有多少缺失值 |
| `df["col"]` / `df[["a","b"]]` | 选一列 / 选多列 |
| `df.resample("ME").mean()` | **改变时间频率** |
| `df.dropna()` / `df.ffill()` | 删除缺失 / 用前一个值填充 |
| `df.pct_change()` | 变化率 |
| `df.corr()` | 相关系数矩阵 |

不需要学完整的 Pandas。

In [ ]:
print("形状：", macro_raw.shape)
print("列名：", list(macro_raw.columns))
print()
print("每列的缺失值数量：")
print(macro_raw.isna().sum())

### 3.1 Day 2 的"降频"，Python 怎么做

Day 2 你已经做过：把高频数据降到低频，让不同频率的指标可以放在一起比较。

Python 用 `resample()`。但**它不会替你决定该用 `mean` 还是 `last` 还是 `sum`**：

- **利率**：一个月内的平均水平 → `mean()`
- **销售量**：如果是"当月总量"就用 `sum()`，如果已经是年化折算值就用 `mean()` 或 `last()`
- **指数点位**：通常用月末值 → `last()`

> **Automation ≠ No judgement.** 自动化的是操作，不是判断。

In [ ]:
macro_monthly = pd.DataFrame({
    "US_10Y_Treasury":   macro_raw["US_10Y_Treasury"].resample("ME").mean(),
    "US_30Y_Mortgage":   macro_raw["US_30Y_Mortgage"].resample("ME").mean(),
    "US_New_Home_Sales": macro_raw["US_New_Home_Sales"].resample("ME").last(),
    "US_CPI":            macro_raw["US_CPI"].resample("ME").last(),
})

macro_monthly = macro_monthly.dropna(how="all")
macro_monthly.index.name = "Date"

print("降频后：", macro_monthly.shape)
macro_monthly.tail()

### Mini Try 1（4 分钟）

1. 把 `US_10Y_Treasury` 改成用 `resample("QE").mean()`，看看季度数据长什么样。
2. 用 `pct_change()` 算出新屋销售的月度环比变化率，存成一个新列 `Home_Sales_MoM`。
3. 想一想：为什么房贷利率用 `mean()` 而不是 `last()`？

In [ ]:
# Mini Try 1：在这里练习

---
# Part 4 — 可视化：只学够用的部分

Day 2 已经学过怎么选图。今天学的是怎么用代码自动生成。

| 想表达什么 | 用什么图 |
|---|---|
| 随时间变化 | 折线图 `plot()` |
| 两个变量的关系 | 散点图 `plot.scatter()` |
| 几个对象的对比 | 柱状图 `plot(kind="bar")` |

In [ ]:
ax = macro_monthly[["US_10Y_Treasury", "US_30Y_Mortgage"]].plot(linewidth=2)
ax.set_title("US 10Y Treasury vs 30Y Mortgage Rate")
ax.set_xlabel("")
ax.set_ylabel("Percent (%)")
ax.legend(["10Y Treasury", "30Y Mortgage"])
plt.tight_layout()
plt.show()

### 4.1 画出来之后，才开始分析

不要停在"图画出来了"。继续问：

- 两条线是不是一起动的？
- 哪一条动得更早？
- 有没有哪一段明显背离？
- 利差（房贷利率 − 国债收益率）稳定吗？

In [ ]:
# 加一列利差，看看它是不是稳定的
macro_monthly["Spread"] = (macro_monthly["US_30Y_Mortgage"]
                           - macro_monthly["US_10Y_Treasury"])

ax = macro_monthly["Spread"].plot(linewidth=2, color="darkorange")
ax.set_title("Mortgage Spread (30Y Mortgage - 10Y Treasury)")
ax.set_xlabel("")
ax.set_ylabel("Percentage points")
plt.tight_layout()
plt.show()

print(macro_monthly["Spread"].describe().round(2))

### 4.2 相关系数：一个快速的描述性检查

In [ ]:
corr_cols = ["US_10Y_Treasury", "US_30Y_Mortgage", "US_New_Home_Sales", "US_CPI"]
corr_table = macro_monthly[corr_cols].corr().round(3)
corr_table

> **Correlation ≠ Causation.**
>
> 相关系数只告诉你"两个数字一起动"，不告诉你"谁导致了谁"。
> 房贷利率和新屋销售负相关，符合 Day 2 的逻辑；但真正的因果链需要经济学解释，不是这张表给的。

In [ ]:
plot_df = macro_monthly[["US_30Y_Mortgage", "US_New_Home_Sales"]].dropna()

ax = plot_df.plot.scatter(x="US_30Y_Mortgage", y="US_New_Home_Sales", alpha=0.6)
ax.set_title("Mortgage Rate vs New Home Sales")
ax.set_xlabel("30Y Mortgage Rate (%)")
ax.set_ylabel("New Home Sales (thousands, SAAR)")
plt.tight_layout()
plt.show()

r = plot_df.corr().iloc[0, 1]
print(f"相关系数：{r:.3f}")

### Mini Try 2（5 分钟）

1. 画一张 `US_CPI` 的折线图，标题写成 "US CPI"。
2. 算出 `US_10Y_Treasury` 和 `US_New_Home_Sales` 的相关系数。
3. 用一句话写下你的观察 —— 这句话下午要放进 README 的 Findings。

In [ ]:
# Mini Try 2：在这里练习

---
# Part 5 — 股价：从宏观逻辑进入公司

宏观逻辑讲完之后，投资者会问：**那这两家公司的股票表现怎么样？**

- **EDB / FRED** 回答：宏观经济发生了什么？（指标 × 时间）
- **WSD** 回答：这只证券过去怎么变化？（证券 × 时间）

今天只用收盘价。

In [ ]:
def _demo_prices(codes, start, end):
    rng = np.random.default_rng(6692285)
    dates = pd.bdate_range(start, end)
    out = {}
    for i, c in enumerate(codes):
        drift = 0.0002 + i * 0.00005
        steps = rng.normal(drift, 0.018, len(dates))
        out[c] = 60 * np.exp(np.cumsum(steps)) * (1 + 0.4 * i)
    return pd.DataFrame(out, index=dates)


def load_prices(codes, start, end):
    if DATA_SOURCE == "wind" and WIND_AVAILABLE:
        r = w.wsd(",".join(codes), "close", start, end, "")
        if r.ErrorCode == 0:
            df = pd.DataFrame(dict(zip(r.Codes, r.Data)),
                              index=pd.to_datetime(r.Times))
            print("[WIND] 股价获取成功")
            return df.sort_index()
        print("[WIND] 获取失败，回退到 demo")
    print("[DEMO] 使用模拟股价")
    return _demo_prices(codes, start, end)


market_prices = load_prices(SECURITIES, START_DATE, END_DATE)
market_prices = market_prices.ffill().dropna(how="all")
market_prices.index.name = "Date"
market_prices.tail()

### 5.1 Normalized Performance：把起点都设为 100

两只股票的价格水平不一样（比如一只 80 港元，一只 25 港元），直接画在一起没法比较。

标准做法：**每条价格序列都除以自己的第一个值，再乘以 100。**

这样所有线都从 100 出发，之后的数字就直接是"相对起点涨跌了百分之多少"。
`120` 就是涨了 20%。

In [ ]:
normalized = market_prices / market_prices.iloc[0] * 100
normalized.index.name = "Date"

ax = normalized.plot(linewidth=2)
ax.set_title("Normalized Share Price Performance (Start = 100)")
ax.set_xlabel("")
ax.set_ylabel("Index (Start = 100)")
ax.legend([SEC_LABELS.get(c, c) for c in normalized.columns])
ax.axhline(100, color="grey", linewidth=1, linestyle="--")
plt.tight_layout()
plt.show()

print(normalized.tail(1).round(1))

### Mini Try 3（4 分钟）

1. 用 `pct_change()` 算出两只股票的日收益率。
2. 用 `.std() * (252 ** 0.5)` 算出年化波动率，比较哪一只波动更大。
3. 试着把 `normalized` 改成从 2020 年开始（提示：`normalized.loc["2020":]` 之后要重新除以第一行）。

In [ ]:
# Mini Try 3：在这里练习
daily_returns = market_prices.pct_change()
daily_returns.head()

---
# Part 6 — 公司截面数据（Snapshot）

- **WSD** 回答：这家公司**过去**怎么变化？（时间序列）
- **WSS** 回答：这些公司**现在**有什么不同？（截面）

截面数据的形状是 **公司 × 指标**，没有时间轴。

In [ ]:
def load_snapshot(codes, fields, labels):
    if DATA_SOURCE == "wind" and WIND_AVAILABLE:
        r = w.wss(",".join(codes), ",".join(fields), "")
        if r.ErrorCode == 0:
            df = pd.DataFrame(dict(zip(labels, r.Data)), index=r.Codes)
            print("[WIND] 截面数据获取成功")
            return df
        print("[WIND] 获取失败，回退到 demo")
    print("[DEMO] 使用模拟截面数据")
    return pd.DataFrame(
        {labels[0]: [18.2, 22.5], labels[1]: [3.1, 2.4], labels[2]: [1580.0, 420.0]},
        index=codes)


company_snapshot = load_snapshot(SECURITIES, WSS_FIELDS, WSS_LABELS)
company_snapshot.index.name = "Security"
company_snapshot["Name"] = [SEC_LABELS.get(c, c) for c in company_snapshot.index]
company_snapshot

In [ ]:
ax = company_snapshot[["PE(TTM)", "PB(MRQ)"]].plot(kind="bar")
ax.set_title("Company Snapshot: Valuation Multiples")
ax.set_xlabel("")
ax.set_ylabel("Multiple (x)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
# Part 7 — 三种数据结构的心智模型

拿到一个问题，**先问自己需要哪一种形状的数据**：

| 数据形状 | 接口 | 它回答的问题 |
|---|---|---|
| 指标 × 时间 | EDB / FRED | 宏观经济发生了什么？ |
| 证券 × 时间 | WSD | 这只证券过去怎么变化？ |
| 公司 × 指标 | WSS | 这些公司现在有什么不同？ |

### 小测验

下面三个问题，各需要哪一种？

1. "过去五年中国 CPI 怎么走的？"
2. "创科实业和泉峰控股哪家估值更贵？"
3. "0669.HK 过去三年的最大回撤是多少？"

<details><summary>点开看答案</summary>

1. 指标 × 时间（EDB / FRED）　2. 公司 × 指标（WSS）　3. 证券 × 时间（WSD）

</details>

---
# Part 8 — 导出数据：下午产品的起点
**这一部分是今天上午和下午之间的桥。**

## 为什么要导出

一个真正的分析产品，**取数**和**展示**必须分开：

- 你的 Streamlit 应用不应该每次打开都去连一次数据库
- 别人打开你的 GitHub 仓库时，不需要有 Wind 账号也能看到结果
- 数据存成 CSV，图表就可以随时重画，代码改了也不用重新取数

> **分析师的习惯：算一次，存下来，画很多次。**

## 导出什么

下面会生成六个文件。前五个是数据，最后一个是**数据字典** —— 它说明每一列是什么、单位是什么、从哪来。
写数据字典只要五分钟，但这是区分"作业"和"作品"的地方。

In [ ]:
EXPORT_DIR = Path("day5_output")
EXPORT_DIR.mkdir(exist_ok=True)

# 1. 宏观月度数据
macro_monthly.to_csv(EXPORT_DIR / "macro_monthly.csv", index_label="Date")

# 2. 股价原始收盘价（app 里可以再算收益率）
market_prices.to_csv(EXPORT_DIR / "market_prices.csv", index_label="Date")

# 3. 归一化表现（起点 = 100）
normalized.to_csv(EXPORT_DIR / "market_normalized.csv", index_label="Date")

# 4. 公司截面
company_snapshot.to_csv(EXPORT_DIR / "company_snapshot.csv", index_label="Security")

# 5. 相关系数表
corr_table.to_csv(EXPORT_DIR / "correlation.csv", index_label="Indicator")

print("已导出：")
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size:>8,} bytes")

In [ ]:
# 6. 数据字典 —— 告诉别人（和三个月后的自己）每一列是什么
data_dictionary = pd.DataFrame([
    {"file": "macro_monthly.csv", "column": "US_10Y_Treasury",
     "description": "美国 10 年期国债收益率，月度平均", "unit": "%",
     "frequency": "Monthly", "source": "FRED: DGS10"},
    {"file": "macro_monthly.csv", "column": "US_30Y_Mortgage",
     "description": "美国 30 年期固定房贷利率，月度平均", "unit": "%",
     "frequency": "Monthly", "source": "FRED: MORTGAGE30US"},
    {"file": "macro_monthly.csv", "column": "US_New_Home_Sales",
     "description": "美国新建独栋住宅销售，季调年化", "unit": "千套",
     "frequency": "Monthly", "source": "FRED: HSN1F"},
    {"file": "macro_monthly.csv", "column": "US_CPI",
     "description": "美国消费者价格指数（所有城市消费者）", "unit": "Index 1982-84=100",
     "frequency": "Monthly", "source": "FRED: CPIAUCSL"},
    {"file": "macro_monthly.csv", "column": "Spread",
     "description": "房贷利率减国债收益率", "unit": "百分点",
     "frequency": "Monthly", "source": "计算得出"},
    {"file": "market_prices.csv", "column": "0669.HK / 2285.HK",
     "description": "日收盘价", "unit": "HKD",
     "frequency": "Daily", "source": "见 README"},
    {"file": "market_normalized.csv", "column": "0669.HK / 2285.HK",
     "description": "归一化股价表现，期初 = 100", "unit": "Index",
     "frequency": "Daily", "source": "由收盘价计算"},
    {"file": "company_snapshot.csv", "column": "PE(TTM) / PB(MRQ) / Market Cap",
     "description": "估值倍数与市值截面快照", "unit": "倍 / 亿",
     "frequency": "Snapshot", "source": "见 README"},
])

data_dictionary.to_csv(EXPORT_DIR / "data_dictionary.csv",
                       index=False, encoding="utf-8-sig")
print("数据字典已导出，共", len(data_dictionary), "行")
data_dictionary.head()

In [ ]:
# 读回来检查一遍 —— 这一步很重要，下午 app 就是这样读的
check = pd.read_csv(EXPORT_DIR / "macro_monthly.csv",
                    index_col="Date", parse_dates=True)

print("读回来的形状：", check.shape)
print("索引类型：", type(check.index))     # 应该是 DatetimeIndex
print()
check.tail(3)

> ⚠️ 注意 `parse_dates=True`。
>
> 如果不加，日期会被读成**字符串**，画出来的图 x 轴会是一堆文本，排序也会乱。
> 下午的 `streamlit_app.py` 里已经帮你加好了。

---
# Part 9 — 从 Notebook 到数据产品

到这里你已经完成了：

> 数据源 → DataFrame → 清洗 / 变频 → 分析 → 可视化 → **导出**

Notebook 适合**你自己**：分析、试错、留下推理过程。
但你没法把一个 `.ipynb` 发给不懂 Python 的人，让他自己调日期看结果。

**Streamlit** 就是给这套分析加一个界面。核心思想只有一句：

> Notebook 已经把数据算好并存成 CSV 了。Streamlit 只是把 CSV 读出来，画在网页上。

下面这段代码就是下午模板仓库里 `streamlit_app.py` 的简化版：

```python
import streamlit as st
import pandas as pd

st.title("Power Tools Industry Monitor")

macro = pd.read_csv("data/macro_monthly.csv", index_col="Date", parse_dates=True)

st.subheader("Macro Conditions")
st.line_chart(macro[["US_10Y_Treasury", "US_30Y_Mortgage"]])
```

三行代码，就有了一个网页。

**今天不学**：session state、cache、复杂布局、回调函数。

---
# 上午 Takeaway

今天要带走的是一个 **Workflow**，不是一堆函数名：

> **研究问题 → 找数据 → 取数 → DataFrame → 清洗 / 变频 → 分析 → 可视化 → 导出 → 可复用的产品**

三个真正重要的观念：

1. **取数和分析要分开。** 换数据源不该导致重写分析代码。
2. **Automation ≠ No judgement.** `resample` 不会替你决定用 mean 还是 last。
3. **算一次，存下来，画很多次。**

---

# 下午你要交付的东西

一个**公开的 GitHub 仓库**，包含：

| 文件 | 内容 |
|---|---|
| `notebook/analysis.ipynb` | 你的分析过程 |
| `data/*.csv` | 你导出的数据 + 数据字典 |
| `streamlit_app.py` | 你的应用 |
| `requirements.txt` | 依赖清单 |
| `README.md` | 研究问题、数据来源、3–5 条 findings、应用截图 |

外加一个**部署好的 Streamlit 网址**。

### 最低要求
- 至少 2 类数据（宏观 + 公司）
- 至少 2 个处理步骤（变频、归一化、计算变化率……）
- 至少 3 张图
- 3–5 条用文字写出来的 findings

### 为什么这值得做

三个月后你面试，面试官问"你会 Python 吗"，你可以给他一个网址，而不是一句"我学过"。

下午的操作手册见 **`下午操作指南_GitHub_Streamlit.md`**。